# Train Cross-Sell VAE

This notebook trains the `RecommendationAwareVAE` in cross-sell mode on the customer
feature matrix produced by `generate_customer_features.ipynb`.

**What gets saved**

| File | Contents |
|---|---|
| `run_dir/loss_history.csv` | Per-epoch loss breakdown (recon, KL, BCE, BPR) |
| `run_dir/metrics.csv` | Final Recall@K / NDCG@K / MAP@K |
| `run_dir/latent_embeddings.csv` | Latent-mean vectors (one row per customer) |
| `run_dir/model.pt` | Serialised model weights (`torch.save`) |
| `run_dir/scaler.pkl` | Fitted `StandardScaler` (needed for inference) |
| `run_dir/imputer.pkl` | Fitted `SimpleImputer` (needed for inference) |
| `run_dir/config.json` | Training hyperparameters |
| `run_dir/loss_curves.png` | Loss-curve plot |

In [ ]:
import sys
import json
import pickle
import dataclasses
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure src/ is on sys.path so package imports resolve without pip install -e
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from embeddings import (
    TORCH_AVAILABLE,
    RecommendationAwareVAEConfig,
    build_recommendation_targets,
    evaluate_cross_sell_predictions,
    preprocess_feature_matrix,
    train_recommendation_aware_vae,
)

if not TORCH_AVAILABLE:
    raise RuntimeError("PyTorch is required. Install it with: pip install torch")

import torch
print(f"Torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print("Imports OK")

---
## Configuration

Adjust paths and hyperparameters here before running the rest of the notebook.

In [ ]:
# ── Input data ──────────────────────────────────────────────────────────────
# Output of generate_customer_features.ipynb
FEATURE_CSV = Path("C:/Users/dijan/Desktop/customer_features.csv")

# ── Output directory ────────────────────────────────────────────────────────
# A timestamped sub-folder is created so previous runs are never overwritten
BASE_OUTPUT_DIR = PROJECT_ROOT / "output" / "cross_sell_vae"
RUN_TIMESTAMP   = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR         = BASE_OUTPUT_DIR / RUN_TIMESTAMP

# ── VAE hyperparameters ─────────────────────────────────────────────────────
LATENT_DIM           = 24
HIDDEN_DIMS          = (512, 256, 128)
BETA                 = 0.08        # KL weight at full warmup
KL_WARMUP_EPOCHS     = 20          # linearly ramp beta from 0 → BETA over these epochs
EPOCHS               = 120
BATCH_SIZE           = 256
LEARNING_RATE        = 1e-3

# ── Cross-sell settings ─────────────────────────────────────────────────────
CROSS_SELL_BCE_WEIGHT       = 1.0   # weight on the BCE auxiliary head
CROSS_SELL_BPR_WEIGHT       = 0.1   # weight on sampled BPR ranking loss; 0 = off
CROSS_SELL_NEGATIVE_SAMPLES = 4     # negative pairs per positive in BPR
CROSS_SELL_K_VALUES         = (5, 10, 20)

RANDOM_STATE = 42

print(f"Feature CSV : {FEATURE_CSV}")
print(f"Run dir     : {RUN_DIR}")

---
## Step 1 — Load feature matrix

In [ ]:
feature_df = pd.read_csv(FEATURE_CSV, index_col=0)

print(f"Shape      : {feature_df.shape}  ({feature_df.shape[0]:,} customers × {feature_df.shape[1]} features)")
print(f"Null values: {feature_df.isna().sum().sum()}")
feature_df.head(3)

## Step 2 — Standardize and build cross-sell targets

`preprocess_feature_matrix` drops high-missingness columns, median-imputes, then
StandardScaler normalises — producing the float matrix the VAE trains on.

`build_recommendation_targets` scans for `share__Product group__*` columns and
converts them to a binary purchase-indicator matrix used as the BCE target.

In [ ]:
cleaned_df, matrix, imputer, scaler = preprocess_feature_matrix(feature_df)
targets = build_recommendation_targets(cleaned_df)

print(f"Training matrix  : {matrix.shape}")
print(f"Target keys      : {sorted(targets.keys())}")

cross_sell_target = targets.get("cross_sell_product_group_binary")
if cross_sell_target is None:
    raise RuntimeError(
        "No 'cross_sell_product_group_binary' target found. "
        "Make sure the feature matrix contains 'share__Product group__*' columns."
    )

print(f"Cross-sell target: {cross_sell_target.shape}  "
      f"(density = {cross_sell_target.mean():.3f})")

---
## Step 3 — Build config and train

In [ ]:
config = RecommendationAwareVAEConfig(
    loss_variant="cross_sell",
    latent_dim=LATENT_DIM,
    hidden_dims=HIDDEN_DIMS,
    beta=BETA,
    kl_warmup_epochs=KL_WARMUP_EPOCHS,
    auxiliary_weight=1.0,
    cross_sell_target_key="cross_sell_product_group_binary",
    cross_sell_bce_weight=CROSS_SELL_BCE_WEIGHT,
    cross_sell_bpr_weight=CROSS_SELL_BPR_WEIGHT,
    cross_sell_negative_samples=CROSS_SELL_NEGATIVE_SAMPLES,
    cross_sell_k_values=CROSS_SELL_K_VALUES,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    random_state=RANDOM_STATE,
)

print("Starting training …")
result = train_recommendation_aware_vae(
    matrix,
    feature_df=cleaned_df,
    config=config,
)

print("Training complete.")
print(f"  Latent shape : {result.latent_mean.shape}")
print(f"  Final loss   : {result.history['loss_per_row'].iloc[-1]:.4f}")
result.history.tail(5)

---
## Step 4 — Inspect loss curves

In [ ]:
history = result.history.set_index("epoch")

# Identify available per-row columns for plotting
per_row_cols = [c for c in history.columns if c.endswith("_per_row")]

fig, axes = plt.subplots(1, len(per_row_cols), figsize=(4 * len(per_row_cols), 4), sharey=False)
if len(per_row_cols) == 1:
    axes = [axes]

colors = plt.cm.tab10.colors
for ax, col, color in zip(axes, per_row_cols, colors):
    ax.plot(history.index, history[col], color=color)
    ax.set_title(col.replace("_per_row", "").replace("_", " "))
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)

plt.suptitle("Training loss curves — cross-sell VAE", y=1.02)
plt.tight_layout()
plt.show()

---
## Step 5 — Save all artefacts

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)

# 1. Loss history
history_path = RUN_DIR / "loss_history.csv"
result.history.to_csv(history_path, index=False)
print(f"[saved] loss_history   → {history_path}")

# 2. Ranking metrics
metrics_path = RUN_DIR / "metrics.csv"
if result.metrics is not None and not result.metrics.empty:
    result.metrics.to_csv(metrics_path, index=False)
    print(f"[saved] metrics        → {metrics_path}")
    display(result.metrics)
else:
    print("No metrics returned from training.")

# 3. Latent embeddings (customer_id as index)
latent_df = pd.DataFrame(
    result.latent_mean,
    index=cleaned_df.index,
    columns=[f"z_{i}" for i in range(result.latent_mean.shape[1])],
)
latent_path = RUN_DIR / "latent_embeddings.csv"
latent_df.to_csv(latent_path, index=True)
print(f"[saved] latent_emb     → {latent_path}")

# 4. Model weights
model_path = RUN_DIR / "model.pt"
torch.save(result.model.state_dict(), model_path)
print(f"[saved] model weights  → {model_path}")

# 5. Scaler and imputer (needed to transform new data for inference)
scaler_path = RUN_DIR / "scaler.pkl"
imputer_path = RUN_DIR / "imputer.pkl"
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
with open(imputer_path, "wb") as f:
    pickle.dump(imputer, f)
print(f"[saved] scaler         → {scaler_path}")
print(f"[saved] imputer        → {imputer_path}")

# 6. Config
config_path = RUN_DIR / "config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(dataclasses.asdict(config), f, indent=2)
print(f"[saved] config         → {config_path}")

# 7. Loss curve figure
fig, axes = plt.subplots(1, len(per_row_cols), figsize=(4 * len(per_row_cols), 4), sharey=False)
if len(per_row_cols) == 1:
    axes = [axes]
for ax, col, color in zip(axes, per_row_cols, colors):
    ax.plot(history.index, history[col], color=color)
    ax.set_title(col.replace("_per_row", "").replace("_", " "))
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
plt.suptitle("Training loss curves — cross-sell VAE", y=1.02)
plt.tight_layout()
fig_path = RUN_DIR / "loss_curves.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"[saved] loss_curves    → {fig_path}")

print()
print(f"All artefacts written to: {RUN_DIR}")

---
## Step 6 — Quick sanity check on latent space

In [ ]:
from embeddings import evaluate_latent_space

latent_eval = evaluate_latent_space(result.latent_mean, cluster_options=(4, 6, 8, 10, 12))
print("Silhouette scores at different cluster counts:")
display(latent_eval)

# 2-D PCA projection for quick visual inspection
from sklearn.decomposition import PCA

pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
z2d = pca2.fit_transform(result.latent_mean)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(z2d[:, 0], z2d[:, 1], s=4, alpha=0.35, c="steelblue")
ax.set_title("Latent space — PCA projection (2D)")
ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)")
plt.tight_layout()
plt.show()